In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv) 
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

import re
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/google/gemma/transformers/2b-it/2/model.safetensors.index.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/gemma-2b-it.gguf
/kaggle/input/models/google/gemma/transformers/2b-it/2/config.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/model-00001-of-00002.safetensors
/kaggle/input/models/google/gemma/transformers/2b-it/2/model-00002-of-00002.safetensors
/kaggle/input/models/google/gemma/transformers/2b-it/2/tokenizer.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/tokenizer_config.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/special_tokens_map.json
/kaggle/input/models/google/gemma/transformers/2b-it/2/.gitattributes
/kaggle/input/models/google/gemma/transformers/2b-it/2/tokenizer.model
/kaggle/input/models/google/gemma/transformers/2b-it/2/generation_config.json
/kaggle/input/notebooks/davidattah/document-retrieval/submission_retrieval.csv
/kaggle/input/notebooks/davidattah/document-retrieval/__results__.html
/kaggl

In [2]:
DATA_DIR = "/kaggle/input/competitions/african-folktales-slm-challenge"

documents = pd.read_csv(
    os.path.join(DATA_DIR, "documents.csv")
)

train = pd.read_csv(
    os.path.join(DATA_DIR, "train_prompts.csv")
)

test = pd.read_csv(
    os.path.join(DATA_DIR, "test_prompts.csv")
)



In [3]:
DIR = "/kaggle/input/notebooks/davidattah/document-retrieval"
retrieved_df = pd.read_csv(
    os.path.join(DIR, "submission_retrieval.csv")
)






In [4]:


print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [5]:
def create_training_text(row):

    return f"""### Prompt:
{row['prompt']}

### Theme:
{row['theme']}

### Region:
{row['culture_region']}

### Story:
{row['reference_story']}"""



train["text"] = train.apply(
    create_training_text,
    axis=1
)

print(train["text"].iloc[0])

### Prompt:
Explain in story form why the baobab looks upside down.

### Theme:
origin_myth

### Region:
east_africa

### Story:
The first baobab boasted that its roots could drink any star. The soil spirit grew tired of pride and planted the tree head-down so its branches learned humility underground. When travelers rest beneath its wide trunk, they remember: greatness must bow to the place that feeds it.


In [6]:
document_texts = []

for _, row in documents.iterrows():

    text = f"""### Theme:
{row['theme']}

### Region:
{row['culture_region']}

### Folktale:
{row['text']}"""

    document_texts.append(text)

In [7]:
# dataset = Dataset.from_pandas(
#     train[["text"]]
# )

# print(dataset)


prompt_texts = train["text"].tolist()

all_texts = (
    prompt_texts +
    document_texts
)

dataset = Dataset.from_dict({
    "text": all_texts
})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 62
})


In [8]:
MODEL_PATH = "/kaggle/input/models/google/gemma/transformers/2b-it/2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

if torch.cuda.is_available():
    model = model.cuda()

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)




In [10]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 34.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [11]:
model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 19,611,648 || all params: 2,525,784,064 || trainable%: 0.7765


In [12]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [13]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/folktale-lora",

    num_train_epochs=8,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),

    logging_steps=5,

    save_strategy="epoch",

    report_to="none",

    remove_unused_columns=False
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
5,4.827750
10,2.928695
15,2.088909
20,1.274752
25,0.769839
30,0.332686
35,0.216213
40,0.153194
45,0.101696
50,0.095987


TrainOutput(global_step=64, training_loss=1.0161023763939738, metrics={'train_runtime': 55.1044, 'train_samples_per_second': 9.001, 'train_steps_per_second': 1.161, 'total_flos': 505121564663808.0, 'train_loss': 1.0161023763939738, 'epoch': 8.0})

In [15]:
OUTPUT_DIR = "/kaggle/working/folktale-lora"

model.save_pretrained(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

('/kaggle/working/folktale-lora/tokenizer_config.json',
 '/kaggle/working/folktale-lora/chat_template.jinja',
 '/kaggle/working/folktale-lora/tokenizer.json')

In [16]:
def generate_story(
    prompt,
    theme,
    region,
    max_new_tokens=120
):

    text = f"""### Prompt:
{prompt}

### Theme:
{theme}

### Region:
{region}

### Story:
"""

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    if torch.cuda.is_available():
        inputs = {
            k: v.cuda()
            for k, v in inputs.items()
        }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return generated

In [17]:
row = train.iloc[1]

story = generate_story(
    row["prompt"],
    row["theme"],
    row["culture_region"]
)

print(story)

### Prompt:
Tell a story showing generosity warms a communal pot.

### Theme:
moral_tale

### Region:
east_africa

### Story:
A traveler asked for one ladle of stew and was refused by a stingy host. The communal pot in the next compound welcomed him, and steam rose at once. Villagers say generosity warms the stones beneath a meal; greed cools even a full hearth.


In [18]:
def extract_story(text):

    if "### Story:" in text:

        text = text.split(
            "### Story:",
            1
        )[1]

    # Remove accidental sections
    for marker in [
        "### Prompt:",
        "### Theme:",
        "### Region:"
    ]:

        if marker in text:
            text = text.split(
                marker,
                1
            )[0]

    return text.strip()


story = extract_story(story)

print(story)

A traveler asked for one ladle of stew and was refused by a stingy host. The communal pot in the next compound welcomed him, and steam rose at once. Villagers say generosity warms the stones beneath a meal; greed cools even a full hearth.


In [19]:
def generate_candidates(
    prompt,
    theme,
    region,
    n=5
):

    candidates = []

    for _ in range(n):

        story = generate_story(
            prompt,
            theme,
            region
        )

        story = extract_story(story)

        candidates.append(story)

    return candidates



row = test.iloc[1]

candidates = generate_candidates(
    row["prompt"],
    row["theme"],
    row["culture_region"],
    n=5
)

for i, story in enumerate(candidates):

    print("=" * 60)
    print(f"CANDIDATE {i+1}")
    print(story)

CANDIDATE 1
An orphan learned to swim by moonlight and used that skill to save fishermen from a squall. He was honored by the chief and trained to teach children to swim before school began. A storm came, and no one could open a shared pen for a collective dip. The chief called an auction for the pen; every family swam free that day. Villagers say courage is not an island; it is a net.
CANDIDATE 2
The fisherman's net was tangled; no amount of pulling could free it. Yet, the sun was set to paint the waves gold and promised calm. He tied knots with children's braided ropes he had learned from the shore. Together they pulled; and pulled. The net freed at once, and children walked the shore that night with shells as gifts for daylight. Courage, the chief said, is not an absence of fear, but a choice to act in spite of it. He taught others, and the harbor boys braved the squall only families could pay. They stretched the rope that
CANDIDATE 3
An orphan mended nets for a stranger and learned

In [20]:
all_candidates = []

for i, row in test.iterrows():

    prompt_id = row["PromptId"]

    # Retrieved document
    retrieved_story = retrieved_df.iloc[i]["Story"]

    candidates = [
        {
            "source": "retrieval",
            "story": retrieved_story
        }
    ]

    # LoRA generations
    generated = generate_candidates(
        row["prompt"],
        row["theme"],
        row["culture_region"],
        n=15
    )

    for story in generated:

        candidates.append({
            "source": "lora",
            "story": story
        })

    all_candidates.append({
        "PromptId": prompt_id,
        "prompt": row["prompt"],
        "theme": row["theme"],
        "culture_region": row["culture_region"],
        "candidates": candidates
    })

In [21]:
!pip install -q python-Levenshtein
import Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.4 MB/s eta 0:00:00


In [22]:
def get_relevant_references(theme, region):

    relevant = train[
        (train["theme"] == theme) &
        (train["culture_region"] == region)
    ]

    if len(relevant) == 0:

        relevant = train[
            train["theme"] == theme
        ]

    if len(relevant) == 0:

        relevant = train

    return relevant["reference_story"].tolist()

In [23]:
def candidate_score(
    story,
    theme,
    region
):

    references = get_relevant_references(
        theme,
        region
    )

    if not references:
        return 999999

    distances = [
        Levenshtein.distance(
            story,
            reference
        )
        for reference in references
    ]

    return np.mean(distances)

In [24]:
final_rows = []

for item in all_candidates:

    scored = []

    for candidate in item["candidates"]:

        story = candidate["story"]

        score = candidate_score(
            story,
            item["theme"],
            item["culture_region"]
        )

        scored.append({
            "source": candidate["source"],
            "story": story,
            "score": score
        })

    scored.sort(
        key=lambda x: x["score"]
    )

    best = scored[0]

    final_rows.append({
        "PromptId": item["PromptId"],
        "Story": best["story"],
        "source": best["source"],
        "score": best["score"]
    })

In [25]:
final_df = pd.DataFrame(final_rows)

display(final_df)

,PromptId,Story,source,score
0,1001,Returning a lost cowrie shell led the seller t...,lora,125.0
1,1002,An orphan learned to swim by moonlight and use...,lora,219.0
2,1003,Hare borrowed thunder from a hollow log until ...,lora,63.0
3,1004,Hyena saw the moon in a still pond and leapt t...,retrieval,0.0
4,1005,Monkey guarded the hive with honey marks and w...,lora,139.0
5,1006,A star fell to earth each time a girl sang at ...,lora,147.0
6,1007,Long ago the river ran straight and forgot the...,retrieval,0.0
7,1008,Two brothers inherited one path to the grazing...,retrieval,0.0
8,1009,Gratitude on the baobab drum brought gentle ra...,lora,134.0
9,1010,Villagers argued whose son would lead the harv...,retrieval,0.0


In [26]:
for _, row in final_df.iterrows():

    print("=" * 80)
    print("PromptId:", row["PromptId"])
    print("Source:", row["source"])
    print("Score:", row["score"])
    print()
    print(row["Story"])
    print()

PromptId: 1001
Source: lora
Score: 125.0

Returning a lost cowrie shell led the seller to a communal pot and a new horn. He later offered a discount in a storm month and was repaid in kind. Honesty, the merchant said, multiplies in another's cooking pot.

PromptId: 1002
Source: lora
Score: 219.0

An orphan learned to swim by moonlight and used that skill to save fishermen from a squall. He was honored by the chief and trained to teach children to swim. When a storm came, parents cast nets while children learned to swim. The chief spoke: practice is a sail folded until the wind needs it.

PromptId: 1003
Source: lora
Score: 63.0

Hare borrowed thunder from a hollow log until Elephant exposed him and thunder returned to the drum. The drum still cracks every time a lie escapes a tongue.

PromptId: 1004
Source: retrieval
Score: 0.0

Hyena saw the moon in a still pond and leapt to seize it, soaking his muzzle in mud. Owl hooted that some lights are for watching, not eating. Hyena laughed, wa

In [27]:
submission = final_df[
    ["PromptId", "Story"]
].copy()

In [28]:
submission_path = (
    "/kaggle/working/submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print(
    f"Submission saved to: {submission_path}"
)

Submission saved to: /kaggle/working/submission.csv
